# Customer Segmentation and Retention Analytics
End-to-end retention project with four serious modeling labs in one notebook.

## 1) Business Problem and Success Criteria
- Objective: reduce avoidable churn under finite retention budget while preserving margin quality.
- Decision lens: prioritize interventions using churn risk, expected value, and operational constraints.
- Success criteria:
  - Primary metric: PR-AUC on holdout
  - Secondary metric: recall at business threshold
  - Tertiary metric: precision at business threshold
  - Calibration: Brier score
- Model governance: select winner through a unified leaderboard and seed-stability checks.

In [ ]:
from pathlib import Path
import json
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.features import (
    FeatureBuildConfig,
    MODEL_FEATURE_COLUMNS,
    SEGMENTATION_FEATURE_COLUMNS,
    add_ltv_proxy,
    build_customer_features,
    build_data_dictionary,
    clean_online_retail,
    load_online_retail,
    run_leakage_checks,
)
from src.modeling import (
    ThresholdConfig,
    benchmark_segmentation_models,
    build_action_policy,
    build_baseline_row,
    build_unified_leaderboard,
    label_segments_with_business_names,
    rerun_top_candidates_across_seeds,
    run_flaml_optimization_lab,
    run_lazypredict_discovery_lab,
    run_manual_engineering_lab,
    run_pycaret_experiment_lab,
    segment_kpi_summary,
    split_train_valid_test,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

PROJECT_NAME = 'customer-segmentation-retention'
ROOT = Path('.')
DATA_DIR = ROOT / 'data'
ARTIFACT_DIR = ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

BUSINESS_ASSUMPTIONS = {
    'retention_offer_cost': 12.0,
    'prevented_churn_value': 80.0,
    'retention_success_rate': 0.30,
    'max_target_share': 0.40,
}
pd.Series(BUSINESS_ASSUMPTIONS)

## 2) Dataset Access and Data Dictionary

In [ ]:
retail_path = DATA_DIR / 'raw' / 'retail' / 'OnlineRetail.csv'
telco_path = DATA_DIR / 'raw' / 'telco' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv'

assert retail_path.exists(), f'Missing dataset: {retail_path}'
print('Retail dataset:', retail_path)
print('Telco reference available:', telco_path.exists())

transactions_raw = load_online_retail(str(retail_path))
print('Raw shape:', transactions_raw.shape)
display(build_data_dictionary(transactions_raw).head(20))

## 3) Data Cleaning and Leakage Checks

In [ ]:
transactions = clean_online_retail(transactions_raw)

feature_config = FeatureBuildConfig(
    history_window_days=365,
    churn_horizon_days=90,
    min_orders=2,
    min_customer_tenure_days=30,
)
snapshot_date = transactions['InvoiceDate'].max() - pd.Timedelta(days=feature_config.churn_horizon_days)
leakage_report = run_leakage_checks(transactions, snapshot_date=snapshot_date, config=feature_config)

print('Clean shape:', transactions.shape)
display(leakage_report)

## 4) Feature Engineering

In [ ]:
customer_features = build_customer_features(
    transactions,
    config=feature_config,
    snapshot_date=snapshot_date,
)
customer_features = add_ltv_proxy(customer_features)

# Keep project theme: segment-aware retention context
seg_leaderboard, seg_registry = benchmark_segmentation_models(
    customer_features,
    feature_cols=SEGMENTATION_FEATURE_COLUMNS,
    project_name=PROJECT_NAME,
    min_clusters=2,
    max_clusters=6,
)
best_seg_model = seg_leaderboard.sort_values('final_rank').iloc[0]['model_name']
customer_features['segment_id'] = seg_registry[best_seg_model]['labels']
customer_features, segment_profile = label_segments_with_business_names(customer_features, segment_col='segment_id')

seg_leaderboard.to_csv(ARTIFACT_DIR / 'leaderboard_segmentation.csv', index=False)

print('Customer base size:', len(customer_features))
print('Churn rate:', round(customer_features['churn'].mean(), 4))
print('Best segment model:', best_seg_model)
display(segment_profile)

In [ ]:
plt.figure(figsize=(10, 4))
sns.countplot(data=customer_features, x='segment_label', order=customer_features['segment_label'].value_counts().index)
plt.xticks(rotation=25, ha='right')
plt.title('Customers by Segment Label')
plt.tight_layout()

## 5) Validation Strategy

In [ ]:
model_features = MODEL_FEATURE_COLUMNS + ['primary_country', 'segment_label']
model_base = customer_features[model_features + ['churn']].copy()

X_train, X_valid, X_test, y_train, y_valid, y_test = split_train_valid_test(
    model_base,
    target_col='churn',
    test_size=0.20,
    valid_size=0.25,
    random_state=SEED,
)

split_report = pd.DataFrame({
    'split': ['train', 'valid', 'test'],
    'rows': [len(X_train), len(X_valid), len(X_test)],
    'churn_rate': [y_train.mean(), y_valid.mean(), y_test.mean()],
})
display(split_report)

threshold_config = ThresholdConfig(
    retention_offer_cost=BUSINESS_ASSUMPTIONS['retention_offer_cost'],
    prevented_churn_value=BUSINESS_ASSUMPTIONS['prevented_churn_value'],
    retention_success_rate=BUSINESS_ASSUMPTIONS['retention_success_rate'],
    max_target_share=BUSINESS_ASSUMPTIONS['max_target_share'],
)

## 6) LazyPredict Discovery Lab

In [ ]:
lazy_benchmark, lazy_eligible, top_families, lazy_retrainers, lazy_details = run_lazypredict_discovery_lab(
    X_train,
    y_train,
    X_valid,
    y_valid,
    X_test,
    y_test,
    threshold_config=threshold_config,
    project_name=PROJECT_NAME,
    top_n_manual_families=3,
    max_models=20,
)

display(lazy_benchmark.head(15))

LazyPredict is used here as a discovery lab, not as the final winner.
Only eligible model families are allowed to move into manual engineering.

## 7) Selection of Top 3 Eligible Models

In [ ]:
eligible_selection = (
    lazy_eligible[['model_name', 'manual_family', 'holdout_primary_metric', 'holdout_secondary_metric', 'final_rank']]
    .drop_duplicates('manual_family')
    .sort_values('final_rank')
    .head(3)
)

print('Selected top families for manual lab:', top_families)
display(eligible_selection)

## 8) Manual Engineering Lab

In [ ]:
manual_results, manual_details, manual_retrainers, manual_threshold_tables = run_manual_engineering_lab(
    manual_families=top_families,
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test,
    threshold_config=threshold_config,
    project_name=PROJECT_NAME,
)

display(manual_results.sort_values('holdout_primary_metric', ascending=False))

In [ ]:
manual_best_name = manual_results.sort_values('holdout_primary_metric', ascending=False).iloc[0]['model_name'] if not manual_results.empty else None
if manual_best_name is not None:
    print('Best manual candidate:', manual_best_name)
    print('Saved artifact:', manual_details[manual_best_name]['artifact_path'])
    display(manual_details[manual_best_name]['threshold_table'].head(10))
    display(manual_details[manual_best_name]['calibration_table'].head(10))
    display(manual_details[manual_best_name]['error_table'])

## 9) FLAML Optimization Lab

In [ ]:
flaml_result, flaml_details, flaml_retrainers, flaml_threshold_tables = run_flaml_optimization_lab(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test,
    threshold_config=threshold_config,
    project_name=PROJECT_NAME,
    time_budget_sec=180,
)

display(flaml_result)
display(pd.DataFrame({
    'item': ['searched_estimators', 'best_estimator', 'best_loss', 'time_budget_sec'],
    'value': [
        ', '.join(flaml_details['searched_estimators']),
        flaml_details['best_estimator'],
        flaml_details['best_loss'],
        flaml_details['time_budget_sec'],
    ],
}))
display(pd.DataFrame([flaml_details['best_config']]))

FLAML conclusion check:
- Compare FLAML holdout PR-AUC and train-time against best manual model to decide if the search budget yielded practical uplift.

## 10) PyCaret Experiment Lab

In [ ]:
family_to_pycaret = {
    'logistic_regression': 'lr',
    'ridge_classifier': 'ridge',
    'random_forest': 'rf',
    'extra_trees': 'et',
    'gradient_boosting': 'gbc',
    'adaboost': 'ada',
    'decision_tree': 'dt',
    'knn': 'knn',
    'xgboost': 'xgboost',
}
pycaret_include = [family_to_pycaret[f] for f in top_families if f in family_to_pycaret]
for default_id in ['lr', 'rf', 'et', 'dt', 'xgboost']:
    if default_id not in pycaret_include:
        pycaret_include.append(default_id)

pycaret_result, pycaret_details, pycaret_retrainers, pycaret_threshold_tables = run_pycaret_experiment_lab(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test,
    threshold_config=threshold_config,
    project_name=PROJECT_NAME,
    include_models=pycaret_include,
    compare_n_select=3,
    tune_iterations=15,
)

display(pycaret_result)
display(pycaret_details['compare_leaderboard'])
display(pycaret_details['tune_metrics'])
if pycaret_details['calibrate_metrics'] is not None:
    display(pycaret_details['calibrate_metrics'])
print('Final PyCaret model id:', pycaret_details['best_model_id'])
print('Saved PyCaret model:', pycaret_details['saved_model_path'])

## 11) Unified Leaderboard and Final Model Ranking

In [ ]:
baseline_result, baseline_retrainers, baseline_threshold_tables = build_baseline_row(
    X_train, y_train, X_valid, y_valid, X_test, y_test,
    threshold_config=threshold_config,
    project_name=PROJECT_NAME,
)

unified_leaderboard = build_unified_leaderboard(
    lazy_benchmark=lazy_benchmark,
    manual_results=manual_results,
    flaml_result=flaml_result,
    pycaret_result=pycaret_result,
    baseline_result=baseline_result,
    lazy_top_n=5,
)

unified_leaderboard.to_csv(ARTIFACT_DIR / 'leaderboard_unified.csv', index=False)
unified_leaderboard.to_csv(ARTIFACT_DIR / 'leaderboard_churn.csv', index=False)

display(unified_leaderboard.head(10))

In [ ]:
all_retrainers = {}
all_retrainers.update(lazy_retrainers)
all_retrainers.update(manual_retrainers)
all_retrainers.update(flaml_retrainers)
all_retrainers.update(pycaret_retrainers)
all_retrainers.update(baseline_retrainers)

seed_stability = rerun_top_candidates_across_seeds(
    unified_leaderboard,
    retrainers=all_retrainers,
    seeds=(42, 77, 2026),
    top_n=3,
)
display(seed_stability)

winner = unified_leaderboard.sort_values('final_rank').iloc[0]
runner_up = unified_leaderboard.sort_values('final_rank').iloc[1]
print('Final winner:', winner['model_name'])
print('Second-best candidate:', runner_up['model_name'])

Ranking rationale:
- Winner is selected by weighted rank score with PR-AUC emphasis and calibration penalty.
- Second-best can be safer when latency/model-size constraints dominate or calibration is materially better.

## 12) Business Recommendation

In [ ]:
winner_name = winner['model_name']
winner_run = all_retrainers[winner_name](SEED)
winner_threshold = float(winner_run['threshold'])

scoring_frame = customer_features.copy()
scoring_frame['churn_score'] = winner_run['score_fn'](scoring_frame[model_features])
policy_df, policy_summary = build_action_policy(
    scoring_frame,
    churn_score_col='churn_score',
    ltv_col='ltv_proxy',
    retention_threshold=winner_threshold,
)
segment_summary = segment_kpi_summary(policy_df)

display(policy_summary)
display(segment_summary)

chosen_threshold_row = winner_run['threshold_table'].iloc[(winner_run['threshold_table']['threshold'] - winner_threshold).abs().argmin()]
print('Approx expected campaign profit per scoring cycle:', round(float(chosen_threshold_row['expected_profit']), 2))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=segment_summary, x='segment_label', y='churn_rate', ax=ax[0])
ax[0].set_title('Segment-Level Churn Rate')
ax[0].tick_params(axis='x', rotation=25)

sns.barplot(data=segment_summary, x='segment_label', y='avg_ltv', ax=ax[1])
ax[1].set_title('Segment-Level LTV Proxy')
ax[1].tick_params(axis='x', rotation=25)
plt.tight_layout()

## 13) Inference / Deployment Path

In [ ]:
deployment_spec = {
    'project_name': PROJECT_NAME,
    'selected_model': winner_name,
    'feature_columns': model_features,
    'threshold': winner_threshold,
    'scoring_contract': {
        'input': 'customer-level feature table',
        'output': ['churn_score', 'action_policy'],
    },
}
with open(ARTIFACT_DIR / 'deployment_spec.json', 'w') as f:
    json.dump(deployment_spec, f, indent=2)

deployment_spec

Suggested deployment sequence:
1. Feature pipeline runs daily/weekly and writes customer-level snapshots.
2. Model scoring service computes `churn_score`.
3. Policy layer maps scores + LTV to action tiers.
4. CRM receives policy cohorts for campaign execution.

## 14) Monitoring / Drift / Retraining Plan

In [ ]:
monitoring_plan = pd.DataFrame({
    'monitor_item': [
        'Input drift (PSI / KS)',
        'Score drift',
        'Calibration drift (Brier)',
        'Business KPI drift (retention conversion, campaign ROI)',
        'Segment mix drift',
    ],
    'cadence': ['weekly', 'weekly', 'bi-weekly', 'monthly', 'weekly'],
    'trigger': [
        'PSI > 0.2 for key features',
        'Score mean shifts > 2 std',
        'Brier worsens > 15%',
        'ROI or conversion drops below agreed floor',
        'Segment share shifts > 10%',
    ],
    'response': [
        'Investigate feature pipeline + retrain candidate',
        'Recalibrate threshold and monitor campaign impact',
        'Run calibration refresh',
        'Adjust targeting rules and offer strategy',
        'Re-run segmentation diagnostics',
    ],
})
display(monitoring_plan)

## 15) Limitations and Next Steps

- Churn label is behavior-defined (inactivity window), not contractual churn.
- LTV is proxy-based and should be replaced with realized margin/contribution when available.
- AutoML track outcomes depend on runtime budget and candidate estimator sets.
- Validation is stratified holdout, not fully temporal CV; future upgrade should include rolling-window validation.

Next steps:
1. Add temporal backtesting and intervention uplift experiments.
2. Add SHAP-based explanation pack for top production candidate.
3. Productionize scoring + monitoring dashboards from `deployment_spec.json`.